# 1D Convolutional Neural Networks: companion notebook

This notebook accompanies `lecture_notes/20_convnets_1D_3D.pdf`, focusing on the **1D convolution** material. It mirrors the style of the 2D CNN companion: short worked examples, shape checks, parameter counts, and compact PyTorch models.

Use it while reading the notes:

- Sections 1-2: what changes when convolution slides over a single axis.
- Section 3: how `nn.Conv1d` expects inputs and how to build a 1D CNN.
- Sections 5-6: choosing 1D convolutions appropriately and debugging common mistakes.


## 0. Setup

The examples use only NumPy arrays and synthetic tensors, so the notebook can run without downloads.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / 'lecture_notes' / '20_convnets_1D_3D.pdf').exists():
            return path
    return start


PROJECT_ROOT = find_project_root()
print('Project root:', PROJECT_ROOT)


## 1. Generalizing convolution from 2D to 1D

Lecture notes, Sections 1 and 2.1.

The convolution pattern stays the same:

- choose a local window;
- take a dot product with a kernel;
- slide the kernel and repeat.

The only change is the geometry. In 1D, the kernel moves along a **single length axis** instead of height and width.


In [ ]:
comparison = [
    ('1D', '(C_in, L)', '(C_out, C_in, k)', '(C_out, L_out)'),
    ('2D', '(C_in, H, W)', '(C_out, C_in, k_h, k_w)', '(C_out, H_out, W_out)'),
    ('3D', '(C_in, D, H, W)', '(C_out, C_in, k_d, k_h, k_w)', '(C_out, D_out, H_out, W_out)'),
]

for name, x_shape, w_shape, z_shape in comparison:
    print(f'{name:>2}  input={x_shape:<22} kernel={w_shape:<30} output={z_shape}')


## 2. A worked 1D convolution example

Lecture notes, Section 2.2.

For a single-channel input sequence `x` and kernel `k`, deep-learning libraries compute **cross-correlation**:

$$
z_j = \sum_{m=0}^{k-1} x_{j+m} \, w_m + b
$$


In [ ]:
def conv1d_numpy(x, kernel, bias=0.0, stride=1, padding=0):
    x = np.asarray(x, dtype=float)
    kernel = np.asarray(kernel, dtype=float)

    if padding > 0:
        x = np.pad(x, pad_width=padding, mode='constant', constant_values=0)

    out_len = (len(x) - len(kernel)) // stride + 1
    out = np.zeros(out_len, dtype=float)

    for j in range(out_len):
        start = j * stride
        window = x[start:start + len(kernel)]
        out[j] = np.sum(window * kernel) + bias
    return out


x = np.array([1, 3, 2, 4, 0, 5], dtype=float)
k = np.array([1, -1, 2], dtype=float)
z = conv1d_numpy(x, k)

print('x =', x)
print('k =', k)
print('z =', z)


In [ ]:
windows = np.stack([x[i:i + len(k)] for i in range(len(z))])
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=False)

axes[0].stem(np.arange(len(x)), x)
axes[0].set_title('Input sequence')
axes[0].set_xlabel('time step')
axes[0].set_ylabel('value')
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(windows, cmap='coolwarm', aspect='auto')
axes[1].set_title('Sliding windows used by the kernel')
axes[1].set_xlabel('kernel position')
axes[1].set_ylabel('output index j')
axes[1].set_xticks(range(len(k)))
for (i, j), value in np.ndenumerate(windows):
    axes[1].text(j, i, f'{value:g}', ha='center', va='center', color='black')
fig.colorbar(im, ax=axes[1], fraction=0.03, pad=0.02)
plt.tight_layout()


### Padding and stride

The output-size formula is exactly the 2D formula applied to one axis:

$$
L_{out} = \left\lfloor 
rac{L + 2p - k}{s} 
ight
floor + 1
$$


In [ ]:
def output_size_1d(length, kernel_size, padding=0, stride=1):
    return (length + 2 * padding - kernel_size) // stride + 1

settings = [
    (60, 5, 0, 1),
    (60, 5, 2, 1),
    (60, 7, 3, 1),
    (60, 5, 0, 2),
    (5000, 7, 3, 1),
]

print('Length  Kernel  Padding  Stride  Output')
for length, kernel_size, padding, stride in settings:
    print(f'{length:>6} {kernel_size:>7} {padding:>8} {stride:>7} {output_size_1d(length, kernel_size, padding, stride):>7}')


### NumPy and PyTorch compute the same result

PyTorch expects 1D-convolution inputs with shape `(N, C_in, L)`.


In [ ]:
conv = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0, bias=True)

with torch.no_grad():
    conv.weight.copy_(torch.tensor(k, dtype=torch.float32).view(1, 1, 3))
    conv.bias.zero_()

torch_input = torch.tensor(x, dtype=torch.float32).view(1, 1, -1)
torch_output = conv(torch_input)

print('Input shape: ', tuple(torch_input.shape))
print('Weight shape:', tuple(conv.weight.shape))
print('Output shape:', tuple(torch_output.shape))
print(torch_output.squeeze().detach().numpy())

np.testing.assert_allclose(z, torch_output.squeeze().detach().numpy())


## 3. Channels and parameter counting

Lecture notes, Section 2.3.

A 1D convolutional layer with `C_in` input channels, `C_out` filters, and kernel size `k` has

$$
C_{out} 	imes (C_{in} 	imes k + 1)
$$

learnable parameters when bias is enabled.


In [ ]:
def conv1d_params(in_channels, out_channels, kernel_size, bias=True):
    weights = out_channels * in_channels * kernel_size
    return weights + (out_channels if bias else 0)

batch = torch.randn(16, 12, 5000)
layer = nn.Conv1d(12, 64, kernel_size=7, padding=3)
out = layer(batch)

manual_count = conv1d_params(in_channels=12, out_channels=64, kernel_size=7, bias=True)
torch_count = sum(p.numel() for p in layer.parameters())

print('Input shape: ', tuple(batch.shape))
print('Weight shape:', tuple(layer.weight.shape))
print('Output shape:', tuple(out.shape))
print('Manual parameter count:', manual_count)
print('PyTorch parameter count:', torch_count)


### Why 1D CNNs are more efficient than MLPs for long sequences

A long ECG or sensor sequence becomes expensive very quickly if flattened into an MLP.


In [ ]:
def mlp_first_layer_params(sequence_length, channels, hidden_units, bias=True):
    weights = sequence_length * channels * hidden_units
    return weights + (hidden_units if bias else 0)

sequence_length = 5000
channels = 12
hidden_units = 128

mlp_params = mlp_first_layer_params(sequence_length, channels, hidden_units)
cnn_params = conv1d_params(in_channels=12, out_channels=64, kernel_size=7)

print(f'MLP first layer: {mlp_params:,} parameters')
print(f'Conv1d layer:    {cnn_params:,} parameters')
print(f'Ratio MLP/CNN:   {mlp_params / cnn_params:,.1f}x')


## 4. Inductive biases for sequential data

Lecture notes, Section 2.4.

A 1D CNN encodes three assumptions that are often useful for signals and sequences:

- **Local temporal context:** each feature depends on a short window.
- **Parameter sharing over time:** the same detector is reused everywhere.
- **Translation equivariance along time:** shifting the input shifts the activation pattern.


In [ ]:
def pulse_sequence(length=40, pulse_start=10, pulse_width=4):
    x = np.zeros(length, dtype=np.float32)
    x[pulse_start:pulse_start + pulse_width] = 1.0
    return x

edge_kernel = np.array([-1.0, 0.0, 1.0], dtype=np.float32)
seq_a = pulse_sequence(pulse_start=8)
seq_b = pulse_sequence(pulse_start=20)

feat_a = conv1d_numpy(seq_a, edge_kernel, padding=1)
feat_b = conv1d_numpy(seq_b, edge_kernel, padding=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 4), sharex='col')
axes[0, 0].plot(seq_a)
axes[0, 0].set_title('Sequence A')
axes[1, 0].plot(feat_a)
axes[1, 0].set_title('Feature map A')
axes[0, 1].plot(seq_b)
axes[0, 1].set_title('Sequence B (shifted pulse)')
axes[1, 1].plot(feat_b)
axes[1, 1].set_title('Feature map B')
for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
plt.tight_layout()


## 5. A compact 1D CNN for time-series classification

Lecture notes, Sections 2.5 and 3.1.

The standard 1D pattern closely mirrors the 2D CNN pattern:

`Conv1d -> BatchNorm1d -> ReLU -> Pool -> ... -> AdaptiveAvgPool1d -> Linear`


In [ ]:
class CNN1D(nn.Module):
    def __init__(self, in_channels=12, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )
        self.gap = nn.AdaptiveAvgPool1d(output_size=1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


model = CNN1D(in_channels=12, num_classes=5)
print(model)


In [ ]:
def trace_shapes_1d(model, x):
    print('input', tuple(x.shape))
    for name, module in model.features.named_children():
        x = module(x)
        print(f'features[{name}] {module.__class__.__name__:>17}: {tuple(x.shape)}')
    x = model.gap(x)
    print(f'gap         {model.gap.__class__.__name__:>17}: {tuple(x.shape)}')
    for name, module in model.classifier.named_children():
        x = module(x)
        print(f'classifier[{name}] {module.__class__.__name__:>10}: {tuple(x.shape)}')
    return x


dummy = torch.randn(16, 12, 5000)
logits = trace_shapes_1d(model, dummy)
print('
Final logits shape:', tuple(logits.shape))


### Adaptive average pooling makes variable-length inference easy

With `AdaptiveAvgPool1d(1)`, the classifier sees a fixed-size vector even when the sequence length changes.


In [ ]:
for length in [4096, 5000, 7777]:
    x = torch.randn(4, 12, length)
    y = model(x)
    print(f'Input length {length:>4} -> logits shape {tuple(y.shape)}')


## 6. Training mechanics and debugging

Lecture notes, Sections 3 and 6.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

signals = torch.randn(8, 12, 5000)
labels = torch.tensor([0, 1, 2, 3, 4, 0, 1, 2])

model.train()
optimizer.zero_grad()
logits = model(signals)
loss = criterion(logits, labels)
loss.backward()
optimizer.step()

print('Logits shape:', tuple(logits.shape))
print('One training-step loss:', float(loss.detach()))


In [ ]:
def tiny_batch_overfit_1d(steps=30, lr=1e-2):
    tiny_x = torch.randn(8, 12, 512)
    tiny_y = torch.arange(8) % 5
    tiny_model = CNN1D(in_channels=12, num_classes=5)
    optimizer = torch.optim.Adam(tiny_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    losses = []
    tiny_model.train()
    for _ in range(steps):
        optimizer.zero_grad()
        loss = criterion(tiny_model(tiny_x), tiny_y)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))
    return losses

losses = tiny_batch_overfit_1d(steps=25)
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Tiny-batch overfit sanity check (1D CNN)')
plt.grid(True)
print('Initial loss:', losses[0])
print('Final loss:  ', losses[-1])


## 7. Entry-ticket style exercises

1. For an ECG input of shape `(12, 5000)`, compute the output shape and parameter count of `Conv1d(12, 64, kernel_size=7, padding=3)`.
2. Explain why `AdaptiveAvgPool1d(1)` is usually preferable to flattening when different examples may have different sequence lengths.
3. Compare the first-layer parameter count of a flattened MLP with the first `Conv1d` layer used above.
4. Give one task where a 1D CNN is appropriate and one where a recurrent or attention-based model may be more natural.


## 8. Key takeaways

- 1D convolution is the same sliding-dot-product idea applied to a single axis.
- In PyTorch, `Conv1d` expects input shape `(N, C, L)`.
- Parameter sharing makes 1D CNNs dramatically smaller than MLPs on long sequences.
- `AdaptiveAvgPool1d(1)` is a practical way to handle variable-length inputs.
- A 1D CNN is a strong baseline when the relevant patterns are local in time.
